# Lab: Synthetic Control Mechanics

[Book home](../index.md)

Use the R kernel. Keep the supplied `data/` folder beside this notebook. Run cells in order after installing the documented R environment. Data loading is entirely local.

## How To Use This Page

Use this page as the first hands-on SCM lab.

- Keep the [Synthetic Control](https://defenceeconomist.github.io/qedlabs/notes/scm/synthetic-control.html) overview open in another tab.
- Use this lab to understand the mechanics of classical SCM before you move to real case studies.
- Read the construction of the design matrices as part of the design, not as data-wrangling boilerplate.
- End by explaining what the optimizer did and whether the fit looks credible enough to learn from.

The code is shown but not executed when the site is rendered. That keeps the page readable while preserving a runnable workflow for teaching and self-study.

## Training Goal

Use the bundled `synth.data` snapshot to make the mechanics of classical SCM visible without requiring its original distribution package:

1.  define the treated unit and donor pool explicitly
2.  construct the treated and donor design matrices explicitly
3.  estimate non-negative donor weights that sum to one with base R
4.  inspect donor weights and predictor balance
5.  interpret the path plot and gap plot without pretending the toy data prove a substantive claim

## Dataset At A Glance

This lab uses a checksummed snapshot of the `synth.data` toy panel. The data are bundled, so the package that originally distributed them is not required at runtime.

- Purpose: mechanics first, not substantive interpretation
- Data shape: a small panel with one treated unit and a compact donor pool
- Main value: it isolates the optimizer, the matrices, and the output objects without the extra complexity of a real policy application
- Main limit: the dataset is too artificial to teach donor contamination, spillovers, or serious placebo logic

## What To Hand Back

By the end of the lab, you should be able to report:

- which unit is treated and which units enter the donor pool
- how the code produces treated and donor predictor matrices and pre-treatment outcome matrices
- which donors receive meaningful weight
- how closely the weighted donors reproduce the treated unit’s predictors and pre-treatment outcomes
- whether the pre-treatment fit looks good enough to trust the toy example as a demonstration of mechanics

## Step 1: Load Packages And Inspect The Toy Panel

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.8)
data_helpers <- c("data/load-data.R", "../data/load-data.R", "docs/labs/data/load-data.R")
data_helpers <- data_helpers[file.exists(data_helpers)]
if (!length(data_helpers)) stop("Extract the complete lab ZIP, including its data folder, before running.")
source(data_helpers[[1]])

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.8)
required_packages <- c(
  "dplyr",
  "ggplot2",
  "tibble",
  "tidyr"
)

missing_packages <- required_packages[!vapply(
  required_packages,
  requireNamespace,
  logical(1),
  quietly = TRUE
)]

if (length(missing_packages) > 0) {
  stop("Install the documented R environment first; missing: ", paste(missing_packages, collapse=", "), call.=FALSE)
}

invisible(lapply(required_packages, library, character.only = TRUE))

synth.data <- qed_data("synth.data")

synth.data |>
  tibble::as_tibble() |>
  glimpse()

Checkpoint:

- Can you see the unit identifier, time variable, predictors, and outcome?
- Which columns are design inputs and which columns are only labels?

## Step 2: Define The Treated Unit And Donor Pool

The point of this step is not software convenience. It is to make the comparison explicit before any optimization happens.

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.8)
treated_unit <- 7

donor_units <- c(29, 2, 13, 17, 32, 38)

unit_lookup <- synth.data |>
  distinct(unit.num, name) |>
  arrange(unit.num)

treated_name <- unit_lookup |>
  filter(unit.num == treated_unit)

donor_lookup <- unit_lookup |>
  filter(unit.num %in% donor_units)

treated_name
donor_lookup

What to discuss:

- SCM does not remove design judgment; it makes it legible.
- Even in a toy example, you should know who is eligible to act as a donor before you estimate weights.

## Step 3: Build The Design Matrices Explicitly

The design object should make every choice visible: which predictors matter, which pre-treatment periods define fit, and which periods will be plotted later. The code below constructs the same kinds of matrices directly from the panel.

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.8)
predictor_years <- 1984:1989
fit_years <- 1984:1990
plot_years <- 1984:1996

unit_predictors <- function(unit_id) {
  unit_data <- synth.data |>
    filter(unit.num == unit_id)

  c(
    X1 = mean(unit_data$X1[unit_data$year %in% predictor_years]),
    X2 = mean(unit_data$X2[unit_data$year %in% predictor_years]),
    X3 = mean(unit_data$X3[unit_data$year %in% predictor_years]),
    Y_1985 = unit_data$Y[match(1985, unit_data$year)],
    Y_1990 = unit_data$Y[match(1990, unit_data$year)]
  )
}

unit_outcomes <- function(unit_id, years) {
  unit_data <- synth.data |>
    filter(unit.num == unit_id)
  unit_data$Y[match(years, unit_data$year)]
}

X1 <- matrix(unit_predictors(treated_unit), ncol = 1)
X0 <- vapply(donor_units, unit_predictors, numeric(nrow(X1)))
Z1 <- matrix(unit_outcomes(treated_unit, fit_years), ncol = 1)
Z0 <- vapply(
  donor_units,
  unit_outcomes,
  numeric(length(fit_years)),
  years = fit_years
)

rownames(X1) <- rownames(X0) <- names(unit_predictors(treated_unit))
rownames(Z1) <- rownames(Z0) <- paste0("Y_", fit_years)
colnames(X0) <- colnames(Z0) <- donor_units

## Step 4: Inspect The Design Objects

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.8)
dim(X1)
dim(X0)
dim(Z1)
dim(Z0)

X1
X0

Checkpoint:

- `X1` and `X0` contain the predictor targets for the treated unit and donors.
- `Z1` and `Z0` contain the pre-treatment outcome history used to optimize fit.
- The named year vectors record the design choices that a package would otherwise hide inside an object.

## Step 5: Estimate The Synthetic Control

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.8)
simplex_weights <- function(theta) c(theta, 1 - sum(theta))

design_target <- rbind(X1, Z1)
design_donors <- rbind(X0, Z0)
row_scale <- apply(cbind(design_target, design_donors), 1, sd)
row_scale[!is.finite(row_scale) | row_scale == 0] <- 1

scaled_target <- design_target[, 1] / row_scale
scaled_donors <- design_donors / row_scale

loss <- function(theta) {
  weights <- simplex_weights(theta)
  mean((scaled_target - as.numeric(scaled_donors %*% weights))^2)
}

gradient <- function(theta) {
  weights <- simplex_weights(theta)
  residual <- scaled_target - as.numeric(scaled_donors %*% weights)
  donor_differences <- scaled_donors[, -ncol(scaled_donors), drop = FALSE] -
    scaled_donors[, ncol(scaled_donors)]
  -2 * colMeans(donor_differences * residual)
}

constraint_matrix <- rbind(
  diag(length(donor_units) - 1),
  rep(-1, length(donor_units) - 1)
)
constraint_boundary <- c(rep(0, length(donor_units) - 1), -1)

scm_fit <- constrOptim(
  theta = rep(1 / length(donor_units), length(donor_units) - 1),
  f = loss,
  grad = gradient,
  ui = constraint_matrix,
  ci = constraint_boundary,
  method = "BFGS",
  control = list(maxit = 5000, reltol = 1e-12)
)

scm_weights <- simplex_weights(scm_fit$par)

stopifnot(
  scm_fit$convergence == 0,
  all(scm_weights >= 0),
  abs(sum(scm_weights) - 1) < 1e-10
)

This is the estimator step. Base R’s `constrOptim()` enforces non-negative donor weights that sum to one while minimizing standardized pre-treatment mismatch. The final donor weight is one minus the sum of the remaining weights, which removes the equality constraint from the optimiser’s parameter vector.

## Step 6: Read Donor Weights And Predictor Balance

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.8)
donor_weights <- tibble(
  unit.num = donor_units,
  weight = scm_weights
) |>
  left_join(unit_lookup, by = "unit.num") |>
  arrange(desc(weight))

predictor_balance <- tibble(
  predictor = rownames(design_target),
  treated = design_target[, 1],
  synthetic = as.numeric(design_donors %*% scm_weights),
  donor_average = rowMeans(design_donors)
)

donor_weights
predictor_balance

What to look for:

- concentrated weights mean only a few donors carry the synthetic comparison
- the balance table shows where the weighted comparison improves on a simple donor average
- both outputs are part of the causal argument

## Step 7: Summarise Fit Quality

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.8)
fit_summary <- tibble(
  comparison = c("Synthetic control", "Unweighted donor average"),
  standardized_mse = c(
    mean((scaled_target - as.numeric(scaled_donors %*% scm_weights))^2),
    mean((scaled_target - rowMeans(scaled_donors))^2)
  )
)

fit_summary

Checkpoint:

- Does the synthetic unit improve predictor balance relative to a simple donor average?
- Are the non-zero donor weights substantively legible or completely opaque?

## Step 8: Plot The Treated And Synthetic Paths

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5.5)
Y1plot <- unit_outcomes(treated_unit, plot_years)
Y0plot <- vapply(donor_units, unit_outcomes, numeric(length(plot_years)), years = plot_years)

gap_tbl <- tibble(
  year = plot_years,
  treated = Y1plot,
  synthetic = as.numeric(Y0plot %*% scm_weights)
) |>
  mutate(gap = treated - synthetic)

gap_tbl |>
  select(year, treated, synthetic) |>
  tidyr::pivot_longer(-year, names_to = "series", values_to = "outcome") |>
  ggplot(aes(year, outcome, colour = series)) +
  geom_line(linewidth = 1) +
  geom_vline(xintercept = 1990, linetype = 2, colour = "gray40") +
  labs(x = "Year", y = "Outcome", colour = NULL) +
  theme_minimal(base_size = 12)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5.5)
ggplot(gap_tbl, aes(year, gap)) +
  geom_hline(yintercept = 0, linetype = 3, colour = "gray50") +
  geom_vline(xintercept = 1990, linetype = 2, colour = "gray40") +
  geom_line(linewidth = 1) +
  labs(x = "Year", y = "Treated minus synthetic") +
  theme_minimal(base_size = 12)

What to discuss:

- the path plot asks whether the synthetic unit tracks the treated unit before treatment
- the gap plot turns that same information into a treatment-effect style display
- if the pre-treatment gap is wide, the right lesson is usually “design problem” rather than “interesting effect”

## Step 9: Calculate The Gap Series Directly

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.8)
gap_tbl

This step matters because it makes the plotted divergence inspectable as ordinary data rather than as an opaque graphics side effect.

## Final Prompt

Write a short answer to these questions:

1.  What part of SCM became clearer once you saw matrix construction and weight estimation as separate steps?
2.  Which donors actually build the synthetic unit?
3.  Does this toy example teach you substantive causal inference, or mainly the structure of the estimator?

## Next Step

Move next to the [Proposition 99 lab](synthetic-control-proposition-99-lab.ipynb), where the same workflow becomes a real comparative case study with donor exclusions, placebo logic, and stronger design stakes.